# 결측치·이상치 처리 (`04_Missing_Outlier`)

```
01_Window_Definition → 02_Integrated_Table → 03_Landmark25_EDA → [04_Missing_Outlier] → 05_Collinearity_Recheck → 03_Modeling
```

이 노트북은 25일 전체 코호트 통합 정본에서 **모델 대상만 추출하고 데이터 계약을 검증**한 뒤,
결측치 원인 진단과 처리, 이상치 점검을 이어서 진행한다.

- 선행: `03_Landmark25_EDA.ipynb` 12절(결측 원인별 이탈률, `imd_band` 지역별 결측)과 7절(구간별 이탈률의 단조성)
- 2026-09-13 통합 정본의 평가 기회·이월 정의가 바뀌어(`0913_01_assessment_opportunity_banked.md`)
  결측 건수와 해석을 새 정의 기준으로 갱신했다. 노트북 번호도 `03` → `04`로 바뀌었다(EDA 단계 삽입).

## 1. 환경 설정과 통합 정본 로드

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'CSV_files').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '../..').resolve()

DATA_PATH = PROJECT_ROOT / 'CSV_files' / '통합 버전' / 'landmark25_all_cohorts.csv'
KEYS = ['code_module', 'code_presentation', 'id_student']

assert DATA_PATH.exists(), f'통합 정본을 찾을 수 없음: {DATA_PATH}'
all_cohorts = pd.read_csv(DATA_PATH)
print('통합 정본:', DATA_PATH)
print('전체 shape:', all_cohorts.shape)

통합 정본: C:\DA_WorkSpace\Personal_OULAD_Churn_Prediction\CSV_files\통합 버전\landmark25_all_cohorts.csv
전체 shape: (32593, 30)


## 2. 모델 대상 추출

25일 시점 재학 상태를 확인할 수 있는 `eligible_at_25 == 1`이면서
`cohort_status_25 == 'model_eligible'`인 행만 선택한다. 원본 통합 CSV는 변경하지 않는다.

In [2]:
model_df = all_cohorts.loc[
    (all_cohorts['eligible_at_25'] == 1)
    & (all_cohorts['cohort_status_25'] == 'model_eligible')
].copy()

model_df.reset_index(drop=True, inplace=True)
model_df.shape

(27661, 30)

## 3. 데이터 계약 검증

행 수, 복합 키 유일성, 타깃 결측·값 범위와 양성 건수를 확인한다.

In [3]:
assert len(all_cohorts) == 32_593, '전체 코호트 행 수 불일치'
assert all_cohorts.duplicated(KEYS).sum() == 0, '전체 코호트 키 중복 발생'
assert len(model_df) == 27_661, '모델 대상 행 수 불일치'
assert model_df.duplicated(KEYS).sum() == 0, '모델 대상 키 중복 발생'
assert model_df['target_churn_after_25'].notna().all(), '모델 대상 타깃 결측 발생'
assert set(model_df['target_churn_after_25'].unique()) <= {0, 1}, '타깃이 0/1 이외의 값을 포함'
assert model_df['target_churn_after_25'].sum() == 5_247, '양성 건수 불일치'
assert model_df['eligible_at_25'].eq(1).all(), '제외 코호트 혼입'
assert model_df['cohort_status_25'].eq('model_eligible').all(), '코호트 상태 불일치'
assert 'n_banked_25' in model_df.columns, '평가 기회·이월 정의 변경(0913_01) 전 통합 정본'

validation_summary = pd.Series({
    '전체 코호트 행 수': len(all_cohorts),
    '모델 대상 행 수': len(model_df),
    '복합 키 중복': model_df.duplicated(KEYS).sum(),
    '타깃 결측': model_df['target_churn_after_25'].isna().sum(),
    '양성 건수': int(model_df['target_churn_after_25'].sum()),
    '음성 건수': int((model_df['target_churn_after_25'] == 0).sum()),
})
validation_summary

전체 코호트 행 수    32593
모델 대상 행 수     27661
복합 키 중복           0
타깃 결측             0
양성 건수          5247
음성 건수         22414
dtype: int64

## 4. 결측치 진단

`model_df`의 결측 컬럼 5개(`imd_band`, `date_registration`, `avg_score_25`, `avg_submit_delay_25`, `submission_rate_25`)를 대상으로 결측 건수·비율을 확인하고, 각 결측이 구조적 결측(파생 규칙상 필연적으로 발생)인지 일반 결측(원본 데이터의 결측)인지 구분한다.

In [4]:
missing_cols = [
    'imd_band', 'date_registration',
    'avg_score_25', 'avg_submit_delay_25', 'submission_rate_25',
]
missing_summary = pd.DataFrame({
    '결측 건수': model_df[missing_cols].isna().sum(),
    '결측 비율(%)': (model_df[missing_cols].isna().mean() * 100).round(2),
})
missing_summary

,결측 건수,결측 비율(%)
imd_band,1030,3.72
date_registration,7,0.03
avg_score_25,8362,30.23
avg_submit_delay_25,8355,30.20
submission_rate_25,5453,19.71


### 4.1 평가 행동 결측 (`submission_rate_25`, `avg_score_25`, `avg_submit_delay_25`)

세 컬럼 모두 `n_opportunity_25`·`n_submitted_25`에서 파생됐으므로, 결측이 `no_assessment_opportunity_25`(25일까지 마감된 평가 기회 자체가 없음) 플래그 및 `n_submitted_25 == 0`(기회는 있었지만 미제출)과 어떻게 대응하는지 확인한다.

In [5]:
# submission_rate_25: 결측이 '평가 기회 없음'과 정확히 일치하는지 확인
opportunity_crosstab = pd.crosstab(
    model_df['submission_rate_25'].isna().rename('submission_rate_25_결측'),
    model_df['no_assessment_opportunity_25'].rename('no_assessment_opportunity_25'),
)
opportunity_crosstab

no_assessment_opportunity_25,0.0,1.0
submission_rate_25_결측,,
False,22208,0
True,0,5453


In [6]:
# avg_score_25 / avg_submit_delay_25: 결측을 원인별로 분해
# 기회 없음은 다시 '과목 일정상 없음'과 '이월로 없음'으로 나눈다 (정의 변경 0913_01)
no_submit = model_df['n_submitted_25'].eq(0)
missing_reason = pd.Series(
    np.select(
        [
            model_df['avg_score_25'].notna(),
            (model_df['n_opportunity_25'] == 0) & (model_df['n_banked_25'] > 0),
            model_df['n_opportunity_25'] == 0,
            no_submit,
        ],
        ['값 있음', '기회 없음(이월)', '기회 없음(과목 일정)', '기회 있으나 미제출'],
        default='제출했으나 점수 없음',
    ),
    index=model_df.index, name='avg_score_25 상태',
)
score_breakdown = missing_reason.value_counts().rename('행 수').to_frame()
score_breakdown['이탈률'] = model_df.groupby(missing_reason)['target_churn_after_25'].mean().round(3)

delay_breakdown = pd.crosstab(
    model_df['avg_submit_delay_25'].isna().rename('avg_submit_delay_25_결측'),
    no_submit.rename('미제출(n_submitted_25==0)'),
)
print(score_breakdown)
print()
print(delay_breakdown)

                   행 수    이탈률
avg_score_25 상태              
값 있음             19299  0.175
기회 없음(과목 일정)      4990  0.108
기회 있으나 미제출        2902  0.415
기회 없음(이월)          463  0.248
제출했으나 점수 없음          7  0.857

미제출(n_submitted_25==0)  False  True 
avg_submit_delay_25_결측              
False                   19306      0
True                        0   8355


In [7]:
# 예외 확인: 제출은 했는데(n_submitted_25 > 0) avg_score_25가 결측인 행
score_edge_cases = model_df.loc[
    (model_df['n_submitted_25'] > 0) & (model_df['avg_score_25'].isna()),
    ['code_module', 'code_presentation', 'id_student', 'n_opportunity_25',
     'n_submitted_25', 'n_missing_25', 'avg_score_25', 'avg_submit_delay_25'],
]
len(score_edge_cases), score_edge_cases

(7,
       code_module code_presentation  id_student  n_opportunity_25  \
 1500          BBB             2013B      534151               1.0   
 4729          BBB             2014B      606501               1.0   
 6584          BBB             2014J      678578               1.0   
 12272         DDD             2013J      427248               1.0   
 18677         FFF             2013B      174436               1.0   
 19570         FFF             2013B      546164               1.0   
 20159         FFF             2013J      126074               1.0   
 
        n_submitted_25  n_missing_25  avg_score_25  avg_submit_delay_25  
 1500              1.0           0.0           NaN                -12.0  
 4729              1.0           0.0           NaN                 -1.0  
 6584              1.0           0.0           NaN                  0.0  
 12272             1.0           0.0           NaN                  0.0  
 18677             1.0           0.0           NaN              

**해석**

- `submission_rate_25` 결측 5,453건은 `no_assessment_opportunity_25 == 1`(=`n_opportunity_25 == 0`)과 100% 일치한다. 완전한 구조적 결측이다.
  정의 변경(0913_01) 전에는 6,766건이었다. BBB 2014J 학생 1,776명에게 평가 기회가 생겨 줄었고, 대신 이월로 기회가 0이 된 재수강생 463명이 더해졌다.
- `avg_score_25` 결측 8,362건의 원인은 네 가지다.

  | 원인 | 행 수 | 이탈률 |
  |---|---:|---:|
  | 기회 없음(과목 일정: EEE·GGG) | 4,990 | 10.8% |
  | 기회 없음(이월) | 463 | 24.8% |
  | 기회 있으나 미제출 | 2,902 | 41.5% |
  | 제출했으나 점수 없음 | 7 | 85.7% |

  앞의 세 원인은 `n_opportunity_25`·`n_banked_25`·`n_submitted_25`로 모델에 이미 전달되는 **구조적 결측**이다.
  **같은 NaN이라도 원인별 이탈률이 4배 가까이 다르므로**, 하나의 값으로 채우면 이 차이가 사라진다(`03_Landmark25_EDA` 12절).
- `avg_submit_delay_25` 결측 8,355건은 `n_submitted_25 == 0`과 100% 일치한다.
- 예외적으로 `avg_score_25`만 **7건**이 제출 기록이 있는데도 결측이다(기존 6건 + BBB 2014J 1건). 0.03%로 전체 결론에 영향이 없고,
  원본 `studentAssessment`의 점수 필드가 비어 있는 개별 사례다.

### 4.2 `imd_band` 결측 — 지역별 분포

`imd_band`(거주 지역 빈곤 지수) 결측이 무작위인지, 특정 지역에 몰려 있는지 지역(`region`)별 결측 비율로 확인한다.
이전 버전은 "IMD가 잉글랜드 지표라 잉글랜드 외 지역에는 값이 없을 것"이라는 가설로 해석했는데,
`03_Landmark25_EDA` 12절에서 이 가설이 데이터와 맞지 않음을 확인했다. 아래에서 잉글랜드 여부를 함께 표시해 다시 검증한다.

In [8]:
ENGLAND = {'East Anglian Region', 'East Midlands Region', 'London Region', 'North Region', 'North Western Region',
           'South East Region', 'South Region', 'South West Region', 'West Midlands Region', 'Yorkshire Region'}
imd_missing_by_region = (
    model_df.groupby('region')['imd_band']
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
    .rename('결측 비율')
    .to_frame()
)
imd_missing_by_region['잉글랜드 여부'] = np.where(imd_missing_by_region.index.isin(ENGLAND), '잉글랜드', '잉글랜드 외')
imd_missing_by_region['결측 비율'] = imd_missing_by_region['결측 비율'].round(4)
print(imd_missing_by_region)
print()
print('결측 여부별 이탈률:', model_df.groupby(model_df['imd_band'].isna())['target_churn_after_25'].mean().round(3).rename({False: '값 있음', True: '결측'}).to_dict())

                       결측 비율 잉글랜드 여부
region                              
North Region          0.4375    잉글랜드
Ireland               0.2262  잉글랜드 외
West Midlands Region  0.0181    잉글랜드
South Region          0.0167    잉글랜드
Scotland              0.0035  잉글랜드 외
Yorkshire Region      0.0024    잉글랜드
South West Region     0.0019    잉글랜드
North Western Region  0.0017    잉글랜드
East Anglian Region   0.0000    잉글랜드
London Region         0.0000    잉글랜드
East Midlands Region  0.0000    잉글랜드
South East Region     0.0000    잉글랜드
Wales                 0.0000  잉글랜드 외

결측 여부별 이탈률: {'값 있음': 0.191, '결측': 0.151}


**해석 (2026-09-13 수정)**

결측이 `North Region`(43.7%)과 `Ireland`(22.6%)에 집중되고 나머지 11개 지역은 2% 미만(대다수 0%)이다.

**이전 해석("IMD가 잉글랜드 지표라 잉글랜드 외 지역에는 값이 없다")은 데이터와 맞지 않아 철회한다.**

- 결측률 1위인 **North Region은 잉글랜드**다.
- 잉글랜드 외 지역인 **Scotland(0.3%)·Wales(0.0%)는 결측이 거의 없다.** Ireland만 가설에 부합한다.

따라서 원인은 **"특정 지역(North Region, Ireland)에 집중된 수집 누락, 원인 불명"**으로 기술한다.
무작위 결측은 아니고, 결측 행의 이탈률(15.1%)이 값 있는 행(19.1%)과 달라 결측 자체가 약한 정보를 가진다.
그러므로 임의 대치보다 **별도 범주(`Unknown`)로 두는 처리는 그대로 타당**하다.

### 4.3 `date_registration` 결측

결측이 27,661건 중 7건(0.03%)으로 극히 적다. 개별 행을 직접 확인한다.

In [9]:
date_registration_missing = model_df.loc[
    model_df['date_registration'].isna(),
    ['code_module', 'code_presentation', 'id_student', 'total_click_25',
     'n_opportunity_25', 'target_churn_after_25'],
]
date_registration_missing

,code_module,code_presentation,id_student,total_click_25,n_opportunity_25,target_churn_after_25
2155,BBB,2013B,630346,0.0,1.0,0.0
10566,CCC,2014J,1777834,0.0,1.0,0.0
11907,DDD,2013B,2707979,0.0,2.0,0.0
11908,DDD,2013B,2710343,0.0,2.0,0.0
14591,DDD,2014B,2710343,0.0,1.0,0.0
16404,EEE,2013J,568751,0.0,0.0,1.0
19992,FFF,2013B,2102658,0.0,1.0,0.0


**해석**

7건은 특정 모듈·학기에 몰려 있지 않고(DDD 3, BBB·CCC·EEE·FFF 각 1), `total_click_25`나 `n_opportunity_25` 등 다른 피처는 정상적으로 채워져 있다.
원본 `studentRegistration`에 등록일 자체가 기록되지 않은 개별 데이터 누락으로 보인다. 이탈은 1건이다.

## 5. 처리 방침

| 컬럼 | 결측 성격 | 방침 |
|---|---|---|
| `submission_rate_25` | 완전한 구조적 결측 (기회 없음과 100% 일치) | NaN 유지. 0으로 채우면 '기회는 있었으나 0% 제출'로 오독된다. (05에서 항등식으로 복원 가능해 최종 피처에서 제외) |
| `avg_score_25`, `avg_submit_delay_25` | 구조적 결측 (기회 없음[과목 일정·이월] + 미제출로 설명, 예외 7건) | NaN 유지. 원인은 `n_opportunity_25`·`n_banked_25`·`n_submitted_25`가 전달한다. 대치 여부는 모델별 파이프라인에서 결정 |
| `imd_band` | 특정 지역(North Region, Ireland)에 집중된 수집 누락, 원인 불명 | `'Unknown'` 범주 추가 |
| `date_registration` | 소수 원본 결측 (7건, 0.03%) | 7행 제외. 배포 시에는 행을 뺄 수 없으므로 모델 파이프라인의 대치기가 같은 상황을 처리한다 |

근거: `work_process/decisions/preprocessing/0912_01_missing_value_policy.md`

## 6. 결측치 처리 적용

5절 방침 중 **데이터셋 단계에서 한 번만 적용하면 되는 두 가지**를 이번 절에서 실제로 반영한다.
나머지 평가 행동 결측(`submission_rate_25`, `avg_score_25`, `avg_submit_delay_25`)은 NaN을
그대로 유지하고, 실제 대치 여부와 방식은 모델별로 다르게 모델링 파이프라인 안에서 처리한다
(근거: `work_process/decisions/preprocessing/0912_01_missing_value_policy.md` 5절).

- `imd_band`: 결측을 `'Unknown'` 범주로 채운다.
- `date_registration`: 결측 7행을 제외한다.

In [10]:
model_df_clean = model_df.copy()

model_df_clean['imd_band'] = model_df_clean['imd_band'].fillna('Unknown')

before_drop = len(model_df_clean)
model_df_clean = model_df_clean.loc[model_df_clean['date_registration'].notna()].copy()
model_df_clean.reset_index(drop=True, inplace=True)
dropped = before_drop - len(model_df_clean)

print(f'imd_band 결측 처리 후 결측 건수: {model_df_clean["imd_band"].isna().sum()}')
print(f'date_registration 결측 제외 행 수: {dropped}')
print(f'처리 후 model_df_clean shape: {model_df_clean.shape}')

imd_band 결측 처리 후 결측 건수: 0
date_registration 결측 제외 행 수: 7
처리 후 model_df_clean shape: (27654, 30)


In [11]:
assert model_df_clean['imd_band'].isna().sum() == 0, 'imd_band 결측 잔존'
assert model_df_clean['date_registration'].isna().sum() == 0, 'date_registration 결측 잔존'
assert model_df_clean.duplicated(KEYS).sum() == 0, '처리 후 키 중복 발생'
assert model_df_clean['target_churn_after_25'].notna().all(), '처리 후 타깃 결측 발생'

remaining_missing = model_df_clean[missing_cols].isna().sum()
print('처리 후 남은 결측(의도적으로 유지, 모델별 파이프라인에서 처리 예정):')
print(remaining_missing)

treatment_summary = pd.Series({
    '처리 전 행 수': len(model_df),
    '처리 후 행 수': len(model_df_clean),
    '제외된 행 수(date_registration 결측)': dropped,
    '처리 전 양성 건수': int(model_df['target_churn_after_25'].sum()),
    '처리 후 양성 건수': int(model_df_clean['target_churn_after_25'].sum()),
})
treatment_summary

처리 후 남은 결측(의도적으로 유지, 모델별 파이프라인에서 처리 예정):
imd_band                  0
date_registration         0
avg_score_25           8355
avg_submit_delay_25    8348
submission_rate_25     5452
dtype: int64


처리 전 행 수                         27661
처리 후 행 수                         27654
제외된 행 수(date_registration 결측)        7
처리 전 양성 건수                        5247
처리 후 양성 건수                        5246
dtype: int64

**해석**

`imd_band` 결측 1,030건은 `'Unknown'` 범주로, `date_registration` 결측 7건은 행 제외로 처리했다(27,661 → 27,654행, 양성 5,247 → 5,246건).
남은 결측은 `avg_score_25` 8,355건, `avg_submit_delay_25` 8,348건, `submission_rate_25` 5,452건이며 의도적으로 유지한다.
이후 모델별 `Pipeline`(Logistic/RF는 대치, XGBoost는 NaN 유지)에서 처리한다.
이후 분석·모델링은 `model_df`가 아니라 `model_df_clean`을 기준으로 진행한다.

### 6.1 정제 데이터셋 저장

`model_df_clean`은 이후 공선성 재검증(`05_Collinearity_Recheck.ipynb`)과 모델링
(`Notebooks/03_Modeling/`)에서 반복 사용된다. 노트북마다 필터링·결측 처리를 다시
구현하면 코드가 중복되고 정의가 어긋날 위험이 있으므로, 이 노트북을 생성 책임자로 두고
CSV로 한 번 저장한다.

- 저장 경로: `CSV_files/통합 버전/model_df_clean_n25.csv`
- 정본은 여전히 `landmark25_all_cohorts.csv`이며, 이 파일은 **이 노트북에서 언제든 재생성
  가능한 파생 산출물**이다.
- 평가 행동 결측 3종(`submission_rate_25`, `avg_score_25`, `avg_submit_delay_25`)은 의도대로
  빈 값으로 저장되고, 다시 읽을 때 `NaN`으로 복원되는지 검증한다.

In [12]:
CLEAN_PATH = PROJECT_ROOT / 'CSV_files' / '통합 버전' / 'model_df_clean_n25.csv'

model_df_clean.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
print('저장 완료:', CLEAN_PATH)

# 라운드트립 검증: 다시 읽어 행 수, 키 유일성, 타깃, 결측 패턴이 그대로인지 확인
reloaded = pd.read_csv(CLEAN_PATH)

assert len(reloaded) == len(model_df_clean), '재로드 행 수 불일치'
assert list(reloaded.columns) == list(model_df_clean.columns), '재로드 컬럼 구성 불일치'
assert reloaded.duplicated(KEYS).sum() == 0, '재로드 키 중복 발생'
assert int(reloaded['target_churn_after_25'].sum()) == int(model_df_clean['target_churn_after_25'].sum()), '재로드 양성 건수 불일치'
assert reloaded['imd_band'].isna().sum() == 0, '재로드 imd_band 결측 발생'
assert reloaded['date_registration'].isna().sum() == 0, '재로드 date_registration 결측 발생'
assert (reloaded[missing_cols].isna().sum() == model_df_clean[missing_cols].isna().sum()).all(), '재로드 결측 패턴 불일치'

print(f'라운드트립 검증 통과 — {len(reloaded):,}행, 양성 {int(reloaded["target_churn_after_25"].sum()):,}건')

저장 완료: C:\DA_WorkSpace\Personal_OULAD_Churn_Prediction\CSV_files\통합 버전\model_df_clean_n25.csv


라운드트립 검증 통과 — 27,654행, 양성 5,246건


## 7. 이상치 점검

`model_df_clean` 기준으로 수치형 피처의 이상치를 점검한다. 대상은 학생 배경 변수
(`num_of_prev_attempts`, `studied_credits`, `date_registration`)와 25일 관측창
행동 변수(`total_click_25`, `active_days_25`, `distinct_resources_25`,
`n_opportunity_25`, `n_submitted_25`, `n_missing_25`, `submission_rate_25`,
`avg_score_25`, `avg_submit_delay_25`)다.

In [13]:
outlier_cols = [
    'num_of_prev_attempts', 'studied_credits', 'date_registration',
    'total_click_25', 'active_days_25', 'distinct_resources_25',
    'n_opportunity_25', 'n_submitted_25', 'n_missing_25',
    'submission_rate_25', 'avg_score_25', 'avg_submit_delay_25',
]

outlier_desc = model_df_clean[outlier_cols].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99, .999]
).T
outlier_desc

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,99.9%,max
num_of_prev_attempts,27654.0,0.159977,0.472751,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.00,4.000,6.0
studied_credits,27654.0,76.350980,38.219574,30.0,30.0,30.0,60.0,60.0,90.0,150.0,210.00,300.000,630.0
date_registration,27654.0,-66.473675,47.517265,-311.0,-200.0,-155.0,-95.0,-53.0,-29.0,-14.0,-2.00,16.000,24.0
total_click_25,27654.0,262.026506,313.025946,0.0,0.0,1.0,63.0,167.0,352.0,830.0,1457.94,2774.022,5818.0
active_days_25,27654.0,9.954654,6.702768,0.0,0.0,1.0,5.0,9.0,15.0,23.0,26.00,26.000,26.0
distinct_resources_25,27654.0,22.488826,16.827738,0.0,0.0,1.0,10.0,19.0,31.0,55.0,73.00,98.000,257.0
n_opportunity_25,27654.0,0.845013,0.464006,0.0,0.0,0.0,1.0,1.0,1.0,1.0,2.00,2.000,2.0
n_submitted_25,27654.0,0.729117,0.509407,0.0,0.0,0.0,0.0,1.0,1.0,1.0,2.00,2.000,2.0
n_missing_25,27654.0,0.115896,0.336733,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.00,2.000,2.0
submission_rate_25,22202.0,0.866003,0.338035,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.00,1.000,1.0


### 7.1 도메인 유효 범위 점검

값 자체가 물리적으로 불가능한지(음수 클릭 수, 0~1 범위를 벗어난 비율 등)를 먼저 확인한다.

In [14]:
domain_checks = {
    'total_click_25 음수': (model_df_clean['total_click_25'] < 0).sum(),
    'active_days_25 범위(0~26) 벗어남': (~model_df_clean['active_days_25'].between(0, 26)).sum(),
    'distinct_resources_25 음수': (model_df_clean['distinct_resources_25'] < 0).sum(),
    'n_opportunity_25 범위(0~2) 벗어남': (~model_df_clean['n_opportunity_25'].between(0, 2)).sum(),
    'n_submitted_25 > n_opportunity_25': (model_df_clean['n_submitted_25'] > model_df_clean['n_opportunity_25']).sum(),
    'n_missing_25 > n_opportunity_25': (model_df_clean['n_missing_25'] > model_df_clean['n_opportunity_25']).sum(),
    'submission_rate_25 범위(0~1) 벗어남': (~model_df_clean['submission_rate_25'].dropna().between(0, 1)).sum(),
    'avg_score_25 범위(0~100) 벗어남': (~model_df_clean['avg_score_25'].dropna().between(0, 100)).sum(),
    'studied_credits <= 0': (model_df_clean['studied_credits'] <= 0).sum(),
    'n_banked_25 음수': (model_df_clean['n_banked_25'] < 0).sum(),
    'n_opportunity_25 + n_banked_25 > 2': ((model_df_clean['n_opportunity_25'] + model_df_clean['n_banked_25']) > 2).sum(),
    'n_banked_25 > 0인데 재수강 이력 0': ((model_df_clean['n_banked_25'] > 0) & (model_df_clean['num_of_prev_attempts'] == 0)).sum(),
}
domain_checks = pd.Series(domain_checks, name='위반 건수')
print(domain_checks)
assert (domain_checks == 0).all(), '도메인 유효성 위반 발생'

total_click_25 음수                     0
active_days_25 범위(0~26) 벗어남           0
distinct_resources_25 음수              0
n_opportunity_25 범위(0~2) 벗어남          0
n_submitted_25 > n_opportunity_25     0
n_missing_25 > n_opportunity_25       0
submission_rate_25 범위(0~1) 벗어남        0
avg_score_25 범위(0~100) 벗어남            0
studied_credits <= 0                  0
n_banked_25 음수                        0
n_opportunity_25 + n_banked_25 > 2    0
n_banked_25 > 0인데 재수강 이력 0            0
Name: 위반 건수, dtype: int64


**해석**

모든 항목이 0건이다(`assert`로 고정). 도메인상 불가능한 값(음수 카운트, 0~1/0~100 범위 밖 비율·점수, 제출·미제출 건수가 기회 건수를 초과하는 논리 모순)은 없다.
정의 변경으로 추가한 이월 관련 점검도 모두 통과했다.

- 이월 수는 음수가 없고, 기회 수 + 이월 수가 과목 일정상 최대 기회(2회)를 넘지 않는다.
- 이월을 받은 학생은 모두 재수강 이력이 있다.

즉 이 데이터셋에는 '명백한 오류성 이상치'가 없고, 남은 것은 극단값(정상 범위 내 드문 값)뿐이다.

### 7.2 IQR 기준 극단값 규모

In [15]:
def iqr_outlier_count(s: pd.Series) -> pd.Series:
    s = s.dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_low = (s < lower).sum()
    n_high = (s > upper).sum()
    return pd.Series({
        'lower_fence': lower, 'upper_fence': upper,
        'n_low': n_low, 'n_high': n_high,
        'n_total_outlier': n_low + n_high,
        'pct_outlier': round((n_low + n_high) / len(s) * 100, 2),
    })

iqr_summary = pd.DataFrame({col: iqr_outlier_count(model_df_clean[col]) for col in outlier_cols}).T
iqr_summary

,lower_fence,upper_fence,n_low,n_high,n_total_outlier,pct_outlier
num_of_prev_attempts,0.0,0.0,0.0,3491.0,3491.0,12.62
studied_credits,15.0,135.0,0.0,1511.0,1511.0,5.46
date_registration,-194.0,70.0,363.0,0.0,363.0,1.31
total_click_25,-370.5,785.5,0.0,1589.0,1589.0,5.75
active_days_25,-10.0,30.0,0.0,0.0,0.0,0.00
distinct_resources_25,-21.5,62.5,0.0,706.0,706.0,2.55
n_opportunity_25,1.0,1.0,5452.0,1166.0,6618.0,23.93
n_submitted_25,-1.5,2.5,0.0,0.0,0.0,0.00
n_missing_25,0.0,0.0,0.0,3054.0,3054.0,11.04
submission_rate_25,1.0,1.0,3054.0,0.0,3054.0,13.76


**해석**

IQR×1.5 기준 극단값 비율이 높게 나오는 컬럼(`total_click_25` 5.75%, `studied_credits` 5.46%, `distinct_resources_25` 2.55%,
`date_registration` 1.31% 등)이 있다. 다만 IQR 기준은 오른쪽으로 치우친(right-skewed) 카운트·활동량 데이터에서 정상 행동까지 이상치로
과다 판정하는 경향이 있다. 실제로 오류인지, 극단적이지만 정상적인 학습 행동인지는 7.3절에서 개별 확인한다.

**주의**: `n_opportunity_25`(23.93%), `n_missing_25`(11.04%), `submission_rate_25`(13.76%), `num_of_prev_attempts`(12.62%)처럼
값이 {0, 1, 2} 근처에 몰린 이산·경계형 컬럼은 IQR이 0이 되어, 정상 값(예: `n_opportunity_25 == 2`, 재수강 1회)까지 기계적으로
"이상치"로 잡힌다. 이 컬럼들은 IQR 방식이 적합하지 않으므로 7.3절 개별 확인에서 제외하고, 7.1절 도메인 유효성 검사(위반 0건)로 충분하다고 판단한다.

`03_Landmark25_EDA` 7절에서 연속형 피처와 이탈률의 관계가 극단 구간에서 뒤집히지 않고 단조로운 것도 확인했다.
극단값이 신호를 왜곡한다는 근거가 없다는 뜻이다.

### 7.3 상위/하위 극단값 개별 확인

In [16]:
top_click = model_df_clean.nlargest(10, 'total_click_25')[
    ['code_module', 'code_presentation', 'id_student', 'total_click_25',
     'active_days_25', 'distinct_resources_25', 'target_churn_after_25']
]
top_click

,code_module,code_presentation,id_student,total_click_25,active_days_25,distinct_resources_25,target_churn_after_25
23852,FFF,2014J,583487,5818.0,26.0,60.0,0.0
21003,FFF,2013J,574315,5501.0,26.0,69.0,0.0
19110,FFF,2013B,497180,4914.0,17.0,40.0,0.0
2368,BBB,2013J,154570,3895.0,26.0,38.0,1.0
7338,CCC,2014B,278861,3717.0,26.0,42.0,1.0
4909,BBB,2014B,621301,3650.0,26.0,28.0,0.0
21877,FFF,2013J,898594,3605.0,26.0,53.0,0.0
21967,FFF,2013J,2239746,3578.0,26.0,41.0,0.0
20370,FFF,2013J,368315,3556.0,26.0,91.0,0.0
25087,FFF,2014J,1871598,3488.0,26.0,55.0,0.0


In [17]:
top_resources = model_df_clean.nlargest(10, 'distinct_resources_25')[
    ['code_module', 'code_presentation', 'id_student', 'distinct_resources_25',
     'total_click_25', 'active_days_25', 'target_churn_after_25']
]
top_resources

,code_module,code_presentation,id_student,distinct_resources_25,total_click_25,active_days_25,target_churn_after_25
12860,DDD,2013J,577196,257.0,1086.0,20.0,1.0
12177,DDD,2013J,373237,159.0,231.0,9.0,0.0
13323,DDD,2013J,610220,133.0,640.0,17.0,0.0
11376,DDD,2013B,536170,128.0,1317.0,26.0,0.0
13434,DDD,2013J,2239650,124.0,1178.0,26.0,0.0
13400,DDD,2013J,1901764,118.0,334.0,7.0,1.0
13713,DDD,2014B,411060,118.0,532.0,17.0,0.0
11468,DDD,2013B,544839,115.0,1045.0,26.0,0.0
11872,DDD,2013B,2624720,114.0,686.0,13.0,0.0
14008,DDD,2014B,598102,113.0,668.0,23.0,0.0


In [18]:
top_credits = model_df_clean.nlargest(10, 'studied_credits')[
    ['code_module', 'code_presentation', 'id_student', 'studied_credits',
     'num_of_prev_attempts', 'target_churn_after_25']
]
top_credits

,code_module,code_presentation,id_student,studied_credits,num_of_prev_attempts,target_churn_after_25
8549,CCC,2014B,1474869,630,0,0.0
23424,FFF,2014J,131597,430,0,0.0
56,AAA,2013J,155550,420,0,0.0
18906,FFF,2013B,400010,360,1,0.0
18996,FFF,2013B,443696,360,1,0.0
21921,FFF,2013J,1935735,360,0,0.0
8639,CCC,2014B,2327710,355,0,1.0
47,AAA,2013J,141377,345,0,1.0
265,AAA,2013J,2065691,330,0,1.0
989,BBB,2013B,395229,330,1,0.0


In [19]:
extreme_registration = model_df_clean.nsmallest(10, 'date_registration')[
    ['code_module', 'code_presentation', 'id_student', 'date_registration',
     'total_click_25', 'target_churn_after_25']
]
extreme_registration

,code_module,code_presentation,id_student,date_registration,total_click_25,target_churn_after_25
7290,CCC,2014B,200372,-311.0,232.0,1.0
7761,CCC,2014B,548854,-310.0,80.0,0.0
10804,DDD,2013B,183714,-310.0,568.0,0.0
7338,CCC,2014B,278861,-305.0,3717.0,1.0
19031,FFF,2013B,476194,-305.0,502.0,0.0
7422,CCC,2014B,371901,-304.0,101.0,0.0
7659,CCC,2014B,521051,-304.0,182.0,0.0
7713,CCC,2014B,536261,-304.0,9.0,0.0
8702,CCC,2014B,2631703,-303.0,220.0,0.0
8533,CCC,2014B,1087710,-302.0,323.0,1.0


In [20]:
delay_tails = pd.concat([
    model_df_clean.nsmallest(5, 'avg_submit_delay_25'),
    model_df_clean.nlargest(5, 'avg_submit_delay_25'),
])[['code_module', 'code_presentation', 'id_student', 'avg_submit_delay_25',
    'n_submitted_25', 'target_churn_after_25']]
delay_tails

,code_module,code_presentation,id_student,avg_submit_delay_25,n_submitted_25,target_churn_after_25
13481,DDD,2013J,2472145,-35.0,1.0,0.0
14537,DDD,2014B,2502516,-32.0,1.0,0.0
12790,DDD,2013J,572735,-31.0,1.0,0.0
12950,DDD,2013J,584914,-31.0,1.0,0.0
13573,DDD,2014B,84576,-31.0,1.0,0.0
4358,BBB,2014B,444221,13.0,1.0,1.0
4579,BBB,2014B,568968,13.0,1.0,0.0
4955,BBB,2014B,623845,13.0,1.0,0.0
5301,BBB,2014B,1431750,13.0,1.0,0.0
5322,BBB,2014B,1782665,13.0,1.0,0.0


**해석**

- `total_click_25`/`distinct_resources_25` 상위값은 짧은 기간에 매우 활발히 학습한
  학생들로, 값 자체가 중복 로그나 수집 오류로 보이는 패턴(예: 동일 타임스탬프 반복,
  음수·비정상 자원 수)은 없다. 극단적이지만 실제 고몰입 행동으로 해석한다.
- `studied_credits` 상위값은 여러 과목을 동시 수강하는 학생으로, OULAD 도메인상 흔한
  패턴이며 오류가 아니다.
- `date_registration` 최솟값(개강 훨씬 이전 등록)도 조기 등록으로 해석 가능한 범위이며,
  같은 학생이 여러 수강에서 반복적으로 나타나는 등 오류를 의심할 패턴은 없다.
- `avg_submit_delay_25` 양쪽 극단(매우 이른 제출/마감 이후 제출)도 `n_submitted_25`와
  모순되지 않는다.

결론적으로 점검한 극단값은 **데이터 수집 오류가 아니라 정상 범위 내의 극단적 학생 행동**
이다. 자세한 처리 방침은 `work_process/decisions/preprocessing/0912_02_outlier_policy.md`
에 확정한다.

## 8. 이상치 처리 방침 (요약)

- 도메인상 불가능한 값이 없으므로 **행 제거는 하지 않는다.**
- 극단값은 실제 신호(고몰입/저몰입 행동)일 가능성이 높으므로 **임의 캡핑(winsorize)도
  하지 않는다.**
- 트리 기반 모델(RandomForest, XGBoost)은 값의 크기가 아니라 순서(분기)만 사용하므로
  극단값에 영향을 받지 않는다. 별도 처리가 필요 없다.
- Logistic Regression처럼 값의 크기(거리)에 민감한 모델은 이상치 제거 대신
  `RobustScaler`(중앙값·IQR 기반 스케일링)를 사용해 극단값의 영향을 완화한다.
  표준화(`StandardScaler`)는 평균·표준편차가 극단값에 민감해 이 데이터셋에는 부적합하다.
- 상세 근거와 모델별 최종 방침은 `work_process/decisions/preprocessing/0912_02_outlier_policy.md`
  참고.

## 다음 작업

결측 5개 컬럼 처리와 이상치 점검(도메인 유효성, IQR 극단값, 개별 확인)까지 완료했다.

다음 노트북은 `05_Collinearity_Recheck.ipynb`이다. 이 노트북이 저장한 `model_df_clean_n25.csv`를 읽어 공선성을 재검증하고 최종 피처 목록을 확정한다.
이번 정의 변경으로 추가된 `n_banked_25`를 피처 후보에 포함해 검증한다.